### imports & set up

In [1]:
import os
import sys
import json
import torch
import functools
import numpy as np
import pandas as pd
import gc
from typing import List
from torch import Tensor
from transformers import AutoTokenizer
from jaxtyping import Int
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import seaborn as sns
import scipy.stats as stats
from scipy.stats import mannwhitneyu, norm
import dotenv
import os
from pathlib import Path
import shutil
import traceback
from huggingface_hub.constants import HF_HUB_CACHE
from transformer_lens.model_bridge import TransformerBridge as TransformerBridge

In [2]:
# Automatically reload modules when they change
%load_ext autoreload
%autoreload 2

In [3]:
print(torch.__version__) #should be 2.6 

2.6.0+cu124


In [4]:
#load environment variables from .env file
if not dotenv.load_dotenv():
    print("Could not find dotenv file...")
else:
    print("Successfully loaded dotenv file. ", end="")
    print(f"Environment PROJECT={os.environ.get('PROJECT')}")

TRANSFORMERLENS_HF_RETRY=1

Successfully loaded dotenv file. Environment PROJECT=my_project


In [5]:
#set up paths
src_path = Path().resolve() / "../src"
sys.path.append(src_path)
project_root = src_path.resolve().parent  
sys.path.append(str(project_root))

In [6]:
#local imports from src 
import src.interp_toolkit as interp_toolkit

In [7]:
# known issue with statsmodels means there is an issue with the following package 
#https://github.com/statsmodels/statsmodels/issues/9584
# import statsmodels.api as sm
# from statsmodels.formula.api import ols

In [8]:
# Check the environment variables...
print(
    f"Hugging face cache inside the dockerfile "
    f"lives at {os.environ.get('HF_HOME')}"
)
secret_token = os.environ.get("HF_TOKEN")
if secret_token is None:
    raise ValueError(
        "Missing HF_TOKEN environment variable. "
        "Set it in your .env file and run load_dotenv()."
    )

redacted = f"{secret_token[:3]}...{secret_token[-3:]}"
print(f"Hugging face token is {redacted}")

Hugging face cache inside the dockerfile lives at /home/aleshins/.cache
Hugging face token is hf_...Xso


In [9]:
torch.set_grad_enabled(False)


### function definitions 

In [10]:
#gpu utils
def test_gpus():
    if torch.cuda.is_available():
        print(f"Number of GPUs available: {torch.cuda.device_count()}")
        for i in range(torch.cuda.device_count()):
            gpu_name = torch.cuda.get_device_name(i)
            print(f"GPU {i}: {gpu_name}")
            # Allocate a tensor on each GPU to test if it's functioning properly
            try:
                x = torch.randn((1000, 1000), device=f"cuda:{i}")
                print(f"Successfully created tensor on GPU {i}")
            except Exception as e:
                print(f"Failed to create tensor on GPU {i}: {e}")
    else:
        print("No GPUs are available.")

def clear_gpu_memory():
    #if GPU memory error, clear the GPU cache. If not working, restart the kernel / juypter notebook
    gc.collect()
    torch.cuda.empty_cache()
    torch.set_grad_enabled(False)

In [11]:

def _ci_lookup(table, model_name):
    if model_name in table:
        return table[model_name]
    for k, v in table.items():
        if k.lower() == model_name.lower():
            return v
    return None


def lookup_chat_template(model_name):
    return prompts.chat_template_for(model_name)


def lookup_num_layers(model_name):
    return _ci_lookup(NUM_LAYERS_BY_MODEL, model_name)


def delete_hf_model_cache(model_path):
    cache_name = "models--" + model_path.replace("/", "--")
    cache_path = os.path.join(HF_HUB_CACHE, cache_name)
    if os.path.exists(cache_path):
        shutil.rmtree(cache_path, ignore_errors=True)
        print(f"  removed disk cache: {cache_path}")
    else:
        print(f"  no disk cache at: {cache_path}")


### Demo

In [34]:
clear_gpu_memory()

In [12]:
#load a model 
model = interp_toolkit.load_model( 'EleutherAI/gpt-j-6B',device="cuda:1")

Loading weights:   0%|          | 0/285 [00:00<?, ?it/s]

Moving model to device:  cuda
Model 'EleutherAI/gpt-j-6B' loaded successfully on cuda:1.


In [27]:
# get hidden activations in response to an input prompt
input_text = "What is your favorite thing?" 
logits,hidden_states = interp_toolkit.get_hidden(model, input_text, layer=0, return_type="residual")

In [26]:
interp_toolkit.generate(model, input_text, max_new_tokens=50)

100%|██████████| 50/50 [00:01<00:00, 31.48it/s]


["<|endoftext|>What is your favorite thing?\nIce cream? chocolate? music?\nOr all! (Lives for a month on pizza boxes.)\n\nThe shop before Pirate Dad\n\nHot stock outside\n\nInside. Things you can get elsewhere, aren't even for sale..."]

### initial demo for mean difference vectors and projections

In [ ]:
#add chat template library functions from vsem
LLAMA_CHAT_TEMPLATE = """<s>[INST] <<SYS>>
{system_prompt}
<</SYS>>

{user_message} [/INST]
"""

#rewrite all of this -- pass the chat template to existing functions, not separate tokenizer
#make a 'prompt constructor' function instead
#read in text from jsons only 
def tokenize_text_llama(
    tokenizer: AutoTokenizer,
    text: List[str]
) -> Int[Tensor, 'batch_size seq_len']:
    prompts = [LLAMA_CHAT_TEMPLATE.format(system_prompt = "You are a helpful assistant.", user_message=msg) for msg in text]
    print(prompts)
    return tokenizer(prompts, padding=True, truncation=False, return_tensors="pt").input_ids




In [ ]:
#work in functionality from vsem

def extract_activation_vector(strings_1, strings_2, model):    

    toks_1 = tokenize_instructions_fn(text=strings_1, tokenizer=model.tokenizer)
    toks_2 = tokenize_instructions_fn(text=strings_2, tokenizer=model.tokenizer)

        
    _, toks_1_cache = model.run_with_cache(toks_1, names_filter=lambda hook_name: 'resid_pre' in hook_name)
    _, toks_2_cache = model.run_with_cache(toks_2, names_filter=lambda hook_name: 'resid_pre' in hook_name)
    
    pos = -1
    num_layers = model.cfg.n_layers
    vector_directions_by_layer = {}
    
    for layer in range(num_layers):

        hook_name = ('resid_pre', layer) 
            
        toks_1_activations = toks_1_cache[hook_name]
        toks_2_activations = toks_2_cache[hook_name]

        toks_1_last = toks_1_activations[:, pos, :] 
        toks_2_last = toks_2_activations[:, pos, :]

        toks_1_mean_act = toks_1_last.mean(dim=0)
        toks_2_mean_act = toks_2_last.mean(dim=0)

        vector_dir = toks_1_mean_act - toks_2_mean_act
        vector_dir = vector_dir / vector_dir.norm()
        vector_directions_by_layer[layer] = vector_dir
    
    del toks_1_cache, toks_2_cache
    gc.collect(); torch.cuda.empty_cache()
    
    return vector_directions_by_layer

In [ ]:
def calculate_projection(model, tokenize_instructions_fn, items, vector_directions_by_layer):

    #get the number of items to project and layers
    num_items = len(items)
    num_layers = len(vector_directions_by_layer)
    print(num_items)
    print(num_layers)

    #initialize an array to hold the projections
    item_projection = np.zeros((num_items, num_layers))
    
    #tokenize the items all at once and send all to the model to obtain their embeddings
    tokens_batch = tokenize_instructions_fn(text=items, tokenizer=model.tokenizer)    
    _, cache = model.run_with_cache(tokens_batch, names_filter=lambda hook_name: 'resid_pre' in hook_name)
    print(cache.keys())
    print(tokens_batch.shape)
    
    #go through each item embedding and calculate the inner product with the vector direction
   
        
    for layer in range(num_layers):

        # get the item embeddings for the current layer using the cache varaible saved above
        hook_name = ('resid_pre', layer)  #this layer activation name
        this_activation = cache[hook_name]          
        print((this_activation.shape))   #should be batch size (num items), length of each batch, and layer size

        for item in range(num_items):

            pos = -1       
            last_token_activation = this_activation[item, pos, :]   #get last token activation, for this batch item          
        
            last_token_activation_norm = last_token_activation / last_token_activation.norm(dim=0, keepdim=True) #normalize
            item_activation_norm = last_token_activation_norm.detach().cpu().numpy()
            #finally, get the vector direction for the current layer
            this_direction = vector_directions_by_layer[layer].detach().cpu().numpy()
            #now project this item onto the vector direction
            item_projection[item, layer] = np.inner(this_direction, item_activation_norm)
    
    return item_projection

In [ ]:
moral_vector_by_layer = extract_activation_vector(morally_positive_adjectives, morally_negative_adjectives,model)

In [ ]:
print(moral_vector_by_layer)

In [ ]:
moral_projections = calculate_projection(
    model=model,
    tokenize_instructions_fn=tokenize_instructions_fn,
    items=stimuli_seong68['item'].tolist(),
    vector_directions_by_layer=moral_vector_by_layer
)

In [ ]:
dd = {
    "llama": """<|begin_of_text|>
<|start_header_id|>system<|end_header_id|>
{system_prompt}
<|eot_id|>
<|start_header_id|>user<|end_header_id|>
{user_prompt}
<|eot_id|>
<|start_header_id|>assistant<|end_header_id|>
"""
}

IndentationError: unexpected indent (1266858786.py, line 1)

###  archive below

In [ ]:
def return_stimuli_seong68(stim_path):
    """
    Returns the list of stimuli from the Seong 68 dataset.
    """
    stim_file = os.path.join(stim_path, 'seong_68.csv')
    stimuli_seong68 = pd.read_csv(stim_file,header=0)
    # stimuli_seong68 = stimuli['item'].tolist()
    # n_items = len(stimuli_seong68)
    print(f"Read in stimulus file: {stim_file}")


    return stimuli_seong68

In [ ]:
stimuli_seong68 = return_stimuli_seong68(stim_path)


In [ ]:
def return_stimuli_dillion(stim_path):
    """
    Returns the list of stimuli from the Dillion dataset.
    """
    # Read in the stimulus file
    stim_file = os.path.join(stim_path, 'dillion_2023.csv')
    stimuli_dillion = pd.read_csv(stim_file,header=0)
    # stimuli_dillion = stimuli['Item'].tolist()
    # n_items = len(stimuli_dillion)
    print(f"Read in stimulus file: {stim_file}")

    return stimuli_dillion

In [ ]:
stimuli_dillion = return_stimuli_dillion(stim_path)
print(stimuli_dillion)

In [ ]:
def return_stimuli_moral_positive(stim_path):
    """
    Returns the list of stimuli from the Moral Positive dataset.
    """
    # Read in the stimulus file
    stim_file = os.path.join(stim_path, 'scenarios_moral_seong_positive.csv')
    stimuli = pd.read_csv(stim_file,header=None)
    stimuli_moral_positive =  stimuli.iloc[:,0].tolist()
    # n_items = len(stimuli_moral_positive)
    print(f"Read in stimulus file: {stim_file}")

    return stimuli_moral_positive
def return_stimuli_moral_negative(stim_path):
    """
    Returns the list of stimuli from the Moral Negative dataset.
    """
    # Read in the stimulus file
    stim_file = os.path.join(stim_path, 'scenarios_moral_seong_negative.csv')
    stimuli = pd.read_csv(stim_file,header=None)
    stimuli_moral_negative =  stimuli.iloc[:,0].tolist()
    # n_items = len(stimuli_moral_negative)
    print(f"Read in stimulus file: {stim_file}")

    return stimuli_moral_negative
    

In [ ]:
scenarios_morally_positive = return_stimuli_moral_positive(stim_path)
scenarios_morally_negative = return_stimuli_moral_negative(stim_path)

In [ ]:
def return_adjectives(stim_path):
    adjfile = os.path.join(stim_path, 'adjectives.json')
    f = open(adjfile, 'r')
    adjectives_dict = json.load(f)
    
    return adjectives_dict

In [ ]:
morally_positive_adjectives = return_adjectives(stim_path)['morality_high']
morally_negative_adjectives = return_adjectives(stim_path)['morality_low']


In [ ]:
morally_positive_adjectives

### archive below

### some basic demos for transfomerlens model interaction

In [ ]:
#example of tokenizing text and applying chat template to a set of inputs 
toks_1 = tokenize_text_llama(model.tokenizer,scenarios_morally_positive)
print(toks_1)
print(toks_1.shape)

In [ ]:
#example of one call to get model activations. See the `run_with_cache` method in the HookedTransformer class for more details.
_, toks_1_cache = model.run_with_cache(toks_1, names_filter=lambda hook_name: 'resid_pre' in hook_name)
print(toks_1_cache.keys())
print(toks_1_cache['blocks.0.hook_resid_pre'].shape) #dimensions are the #batches of input tokens; the size of each batch; and then the layer dimensions